In [ ]:
# rende config.py (in notebooks/) importabile anche dai notebook in sottocartelle
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import concurrent.futures
from concurrent.futures import ProcessPoolExecutor
from scipy import stats
import pandas as pd
import xskillscore as xs
import dask
import logging
from pathlib import Path


In [ ]:
# Configurazione logging
log_file = "04-BIAS_seasons.log"
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s",
                    handlers=[logging.FileHandler(log_file)])


In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var = 'tas'
era_var = '2t'

seasons = ['DJF', 'MAM', 'JJA', 'SON']

DATA_PATH = POST_DATA        # dati multi-anno degli esperimenti
OUTPUT_PATH = FIG_DIR_2025
OBS_PATH = WORK_DIR          # osservazioni + output intermedi


In [ ]:
def bootstrap_quantile_bias(sens, ctrl, ref, iterations=1000):
#    sens = sens.chunk({'member':-1,'time': -1, 'lon': -1, 'lat': -1})
#    ctrl = ctrl.chunk({'member':-1,'time': -1, 'lon': -1, 'lat':-1})
#    ref = ref.chunk({'time': -1, 'lon': -1, 'lat': -1})
    with dask.config.set(**{'array.slicing.split_large_chunks': True}):
        f = xr.concat([sens, ctrl], dim='member').chunk({'member':-1})
        # media su membri E tempo PRIMA di compute: materializza solo (iteration, lat, lon)
        # invece di (iteration, time, lat, lon) -> molta meno RAM (evita OOM/BrokenProcessPool)
        f_ra = xs.resampling.resample_iterations(f, iterations, 'member', replace=True).mean('member').squeeze()
        f_rb = xs.resampling.resample_iterations(f, iterations, 'member', replace=True).mean('member').squeeze()
        bias_f_ra = (f_ra.mean('time') - ref).compute()
        bias_f_rb = (f_rb.mean('time') - ref).compute()
        delta_bias = bias_f_ra - bias_f_rb
        sig_delta = delta_bias.quantile([0.025,0.05,0.10,0.90,0.95,0.975], dim='iteration')
    return sig_delta

In [ ]:
def process_lead_years_season(exp_ctrl, exp_sens, var, y1, y2, season, save_path):
    """Mappe di bias (media membri - ERA5, mediate nel tempo) + t-test per punto
    griglia + bootstrap, per un range di lead year (y1-y2) e una stagione.

    NB — ASSUNZIONE SUI NOMI DEI FILE STAGIONALI (convenzione da 05-Serie_temporale_anomalie; verificare):
      esperimenti: ..._lead_{y1}-{y2}_1x1_ensemble_m{season}_rad.nc
      osservazioni: ERA5_{var}_1x1_{n}{season}.nc  (es. ERA5_tas_1x1_2DJF.nc)
    Se il layout reale è diverso, correggere le due righe di path qui sotto."""
    lead = f"{y1}-{y2}"
    lead_number = y2 - y1 + 1
    logging.info(f"Starting {season} {lead}")
    try:
        # --- esperimenti (multi-anno, stagionali) ---
        dset_ctrl_path = DATA_PATH / exp_ctrl / "1x1" / var / f"{exp_ctrl}_{var}_Amon_EC-Earth3_dcppA-hindcast_lead_{lead}_1x1_ensemble_m{season}_rad.nc"
        dset_ctrl = xr.open_dataset(dset_ctrl_path)
        dset_ctrl['time'] = pd.to_datetime(dset_ctrl['time'].values).normalize().to_period('M').start_time

        dset_sens_path = DATA_PATH / exp_sens / "1x1" / var / f"{exp_sens}_{var}_Amon_EC-Earth3_dcppA-hindcast_lead_{lead}_1x1_ensemble_m{season}_rad.nc"
        dset_sens = xr.open_dataset(dset_sens_path)
        dset_sens['time'] = pd.to_datetime(dset_sens['time'].values).normalize().to_period('M').start_time

        # --- osservazioni (ERA5 stagionale) ---
        obs_path = OBS_PATH / f"ERA5_{var}_1x1_{lead_number}{season}.nc"
        obs = xr.open_dataset(obs_path)
        if lead_number in [2, 4]:
            obs['time'] = pd.to_datetime(obs['time'].values) - pd.DateOffset(months=1)
        obs['time'] = pd.to_datetime(obs['time'].values).normalize().to_period('M').start_time
        obs = obs.rename({f'{era_var}': f'{var}'})
        obs = obs.sel(time=slice(dset_ctrl.time[0], dset_ctrl.time[-1]))

        start_date = '1999-09-01'
        dset_ctrl = dset_ctrl.sel(time=slice(start_date, None))
        dset_sens = dset_sens.sel(time=slice(start_date, None))
        obs = obs.sel(time=slice(start_date, None))

        # --- bias medio nel tempo (mappa lat-lon) ---
        bias_ctrl_ts = (dset_ctrl[var].mean('member') - obs[var])
        bias_ctrl = bias_ctrl_ts.mean('time')
        bias_sens_ts = (dset_sens[var].mean('member') - obs[var])
        bias_sens = bias_sens_ts.mean('time')

        # --- t-test per punto griglia (bias != 0 nel tempo) ---
        logging.info(f"T-Test {season} {lead}")
        _, ctrl_p = stats.ttest_1samp(bias_ctrl_ts, popmean=0, axis=0)
        _, sens_p = stats.ttest_1samp(bias_sens_ts, popmean=0, axis=0)
        ctrl_p = xr.DataArray(ctrl_p, coords=bias_ctrl.coords, dims=["lat", "lon"], name="p")
        sens_p = xr.DataArray(sens_p, coords=bias_sens.coords, dims=["lat", "lon"], name="p")

        # --- bootstrap sulla differenza di bias ---
        logging.info(f"Bootstrap {season} {lead}")
        delta = bootstrap_quantile_bias(dset_ctrl, dset_sens, obs.mean('time'))

        # --- salvataggi (suffisso _{season}) ---
        bias_ctrl.to_dataset(name=var).to_netcdf(f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_bias_{season}.nc')
        bias_sens.to_dataset(name=var).to_netcdf(f'{save_path}/{exp_sens}_{var}_lead_{lead}_bias_{season}.nc')
        ctrl_p.to_netcdf(f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_bias_p_{season}.nc')
        sens_p.to_netcdf(f'{save_path}/{exp_sens}_{var}_lead_{lead}_bias_p_{season}.nc')
        delta.to_netcdf(f'{save_path}/delta_{var}_lead_{lead}_bias_quantile_{season}.nc')
        logging.info(f"file salvati {season} {lead}")
    except Exception as e:
        logging.exception(f"Error {season} {lead}: {e}")


In [ ]:
%%time
# calcolo mappe di bias per tutte le stagioni x tutti i range di lead year
with concurrent.futures.ProcessPoolExecutor(max_workers=2) as executor:
    futures = {}
    for season in seasons:
        for y1 in range(5):
            for y2 in range(y1 + 1, 5):
                fut = executor.submit(process_lead_years_season, exp_ctrl, exp_sens, var, y1, y2, season, OBS_PATH)
                futures[fut] = f"{season} {y1}-{y2}"
    for _i, _f in enumerate(concurrent.futures.as_completed(futures), 1):
        _f.result()
        print(f"  [{futures[_f]}] completato {_i}/{len(futures)}", flush=True)
